In [37]:
# import pytesseract
# from pdf2image import convert_from_path
import re
from openai import OpenAI
import numpy as np

import os
from dotenv import load_dotenv

load_dotenv()  # Loads variables from .env into the system environment
OPEN_AI_KEY = os.getenv("OPEN_AI_KEY")

### Post-OCR cleaning?

In [38]:
with open('output/plain_text/1922/03/mvol-0004-1922-0308.txt','r') as f:
    text = f.readlines()
    text[0] = text[0].replace('¬', '')
    # text.lower

In [39]:
text[0][:500]

'Vol. 20. No. 83. UNIVERSITY OF CHICAGO, WEDNESDAY, MARCH 8, 1922 Price 3 CentsMAROONS IN BIGTEN TUSSLE ATMADISON TONIGHTVarsity Five Closes SeasonWith Two Game SeriesAgainst BadgersBOTH TEAMS IN GREAT SHAPELAST NIGHT’S GYMNASTICSCOREChicago 1183 4/10Minnesota 1116 2/10The last week of tlie Big Ten basketball is at hand, and the Maroonsare facing perhaps the toughest seriesof their entire schedule. Two gamesare carded this week with the Badgers, the first being staged tonight atMadison, and the t'

In [40]:
from wordsegment import load, segment
load() 
' '.join(segment(text[0]))  

'vol20no83 university of chicago wednesday march 81922price3 cents maroons in big ten tussle at madison tonight varsity five closes season with two game series against badgers both teams in great shape last nights gymnastics core chicago 1183410minnesota1116210 the last week of tlie big ten basketball is at hand and the maroons are facing perhaps the toughest series of their entire schedule two games are carded this week with the badgers the first being staged tonight at madison and the two teams appearing saturday in bartlett in the final tussle of the year two crackerjack tilts should result as both teams are going great and both are primed for the series by reason of their thrilling victory over the illini at urbana the maroons are figured to give the cardinals a hard battle and should at least break even in the two games to be staged varsity hits stride norgren s charges really came into their own saturday at urbana up until the playing of that game the varsity five had dropped eve

In [25]:
i = text[0].find("Bobbed hair")
string = text[0][i:11_000]
string

'V'

In [24]:
string

'V'

In [12]:
i = text[0].find("Bobbed hair")
text[0][i:11_000]

'Bobbed hair, the subject of moreyards of news matter than the recentWorld War, again claimed the attention of the University when campus and off-campus authorities gavetheir opinions on the matter in specialstatements to The Daily Maroon yesterday.Tom Eck, coach of the Varsity team,declared in favor of bobbed hair for“smart girls—those who knew enoughto come in out of the rain and whodid not paint their lips too red.” Hesaid that wavy hair was the prerequisite for bobbed locks. Women withhair as straight as that of an Indianshould under no condition have theirhair bobbed.“There are some girls w\'ho are notgood-looking whether their hair isbobbed or not,” he added. “My grandmothers lived to 95 and 96 withouthaving their tresses cut short.”"Doc” Bratfish Gives CommentOnly one out of a hundred womenlook well in bobbed hair is the contention of “Doc” Bratfish, Reynolds clubbarber. However, as far as laboi-saving, convenience and such mattersenter into thee question, every womanshould have

In [13]:
i = text[0].find("Bobbed hair")
text[0][i:11_000]


'Bobbed hair, the subject of moreyards of news matter than the recentWorld War, again claimed the attention of the University when campus and off-campus authorities gavetheir opinions on the matter in specialstatements to The Daily Maroon yesterday.Tom Eck, coach of the Varsity team,declared in favor of bobbed hair for“smart girls—those who knew enoughto come in out of the rain and whodid not paint their lips too red.” Hesaid that wavy hair was the prerequisite for bobbed locks. Women withhair as straight as that of an Indianshould under no condition have theirhair bobbed.“There are some girls w\'ho are notgood-looking whether their hair isbobbed or not,” he added. “My grandmothers lived to 95 and 96 withouthaving their tresses cut short.”"Doc” Bratfish Gives CommentOnly one out of a hundred womenlook well in bobbed hair is the contention of “Doc” Bratfish, Reynolds clubbarber. However, as far as laboi-saving, convenience and such mattersenter into thee question, every womanshould have

### Chunk text

In [15]:
CHUNK_SIZE = 500  # approx words
chunks = []
for page_num, text in enumerate(text):
    words = text.split()
    for i in range(0, len(words), CHUNK_SIZE):
        chunk_text = " ".join(words[i:i+CHUNK_SIZE])
        chunks.append({"page": page_num, "text": chunk_text})

- SQLite FTS5 works best with smaller chunks, e.g., 300–500 tokens per chunk.
- Save metadata: page_num, PDF filename, maybe bounding box info if you want to highlight matches later.


In [36]:
chunks[0]

{'page': 0,
 'text': "Vol. 20. No. 83. UNIVERSITY OF CHICAGO, WEDNESDAY, MARCH 8, 1922 Price 3 CentsMAROONS IN BIGTEN TUSSLE ATMADISON TONIGHTVarsity Five Closes SeasonWith Two Game SeriesAgainst BadgersBOTH TEAMS IN GREAT SHAPELAST NIGHT’S GYMNASTICSCOREChicago 1183 4/10Minnesota 1116 2/10The last week of tlie Big Ten basketball is at hand, and the Maroonsare facing perhaps the toughest seriesof their entire schedule. Two gamesare carded this week with the Badgers, the first being staged tonight atMadison, and the two teams appearing Saturday in Bartlett in the finaltussle of the year.Two cracker-jack tilts should result,as both teams are going great, andboth are primed for the series. Byreason of their thrilling victory overthe Illini at Urbana, the Maroons arefigured to give the Cardinals a hardbattle and should at least break evenin the two games to be staged.Varsity Hits StrideNorgren’s charges really came intotheir own Saturday at Urbana. Upuntil the playing of that game theVarsi

### Generate embeddings

In [16]:
# use openAI / any text embedding model

# client = OpenAI(api_key='${{secrets.OPEN_API_KEY}}')
client = OpenAI(api_key=OPEN_AI_KEY)

for chunk in chunks:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk["text"]
    )
    chunk["embedding"] = response.data[0].embedding

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

•	Save embedding vectors to FAISS for semantic search.
•	For SQLite + FTS5, you don’t strictly need embeddings unless you want vector search inside SQLite (FTS is keyword-based).

In [ ]:
chunks

### Create SQLite FTS5 Table


SQLite FTS5 lets you do full-text search on your chunks:

In [ ]:
import sqlite3

conn = sqlite3.connect("maps_index.db")
c = conn.cursor()

# Create table with FTS5
c.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS documents
USING fts5(
    filename,
    page,
    text
)
""")
conn.commit()

# Insert chunks
for chunk in chunks:
    c.execute(
        "INSERT INTO documents (filename, page, text) VALUES (?, ?, ?)",
        ("old_university_maps.pdf", chunk["page"], chunk["text"])
    )
conn.commit()

So we can do keyword queries:

In [ ]:
query = "campus map"
c.execute("SELECT filename, page, text FROM documents WHERE text MATCH ?", (query,))
results = c.fetchall()

Optional: Combine with Embeddings for Semantic Search
- Keep a FAISS index alongside SQLite.
- Use FTS5 for keyword filtering → FAISS for semantic ranking.
- Example: find pages that mention "library" → rerank using embedding similarity.

In [ ]:
Optional: Metadata

In [ ]:
c.execute("""
CREATE TABLE IF NOT EXISTS metadata (
    filename TEXT,
    page INTEGER,
    embedding BLOB
)
""")

import pickle
for chunk in chunks:
    emb_blob = pickle.dumps(chunk["embedding"])
    c.execute("INSERT INTO metadata (filename, page, embedding) VALUES (?, ?, ?)",
              ("old_university_maps.pdf", chunk["page"], emb_blob))
conn.commit()

- Later, you can retrieve embeddings and compute cosine similarity for semantic RAG queries.


- Save metadata: page_num, PDF filename, maybe bounding box info if you want to highlight matches later.


In [ ]:
# # Google's OCR does NOT do a better job than UChicago's OCR tool

# pages = convert_from_path("pdfs/1902/10/mvol-0004-1902-1001.pdf", dpi=300)
# ocr_texts = []

# for page in pages:
#     text = pytesseract.image_to_string(page)
#     ocr_texts.append(text)

# def clean_text(text):
#     text = text.replace("\n", " ")  # merge lines
#     text = re.sub(r"\s+", " ", text)  # collapse whitespace
#     text = text.strip()
#     return text

# ocr_texts = [clean_text(t) for t in ocr_texts]